# Amazon Bedrock AgentCore Memory for Process Tracking and Analytics 

## Overview

This tutorial demonstrates how to use [AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html) namespaces beyond conversational AI — as state machines for tracking business process lifecycles. You'll use namespaces to represent stages in a credit card recommendation pipeline, move records between stages, and query namespace data for real-time analytics without traditional BI infrastructure.

### Tutorial Details

| Information         | Details                                                          |
|:--------------------|:-----------------------------------------------------------------|
| Tutorial type       | Memory for Process and Analytics                                 |
| Feature             | Long-Term Memory Namespaces                                      |
| Key features        | Namespace Hierarchies, State Transitions, External Data Population |
| Example complexity  | Intermediate                                                     |
| SDK used            | boto3, bedrock-agentcore                                         |

### What You'll Learn

In this tutorial, you'll learn how to:
1. Create a memory instance and populate it externally (not from conversations)
2. Use namespace paths to represent business process states
3. Transition records between namespaces while preserving audit history
4. Query namespace data for product-level analytics
5. Combine namespace queries with LLM analysis for natural language insights
6. View customer-specific journeys across namespace stages

### How It Works

Instead of storing state in database columns, each namespace path represents a stage in the business process:

```
/bank/customers/{id}/recommendations/pending   → awaiting presentation
/bank/customers/{id}/recommendations/shown      → presented to customer
/bank/customers/{id}/recommendations/accepted   → customer interested
/bank/customers/{id}/recommendations/declined   → customer rejected
/bank/customers/{id}/recommendations/applied    → application submitted
```

Moving a record from `pending` to `shown` is a state transition — the namespace path IS the state. No status columns, no joins, no ETL pipelines.

## 0. Prerequisites

To execute this tutorial you will need:
* Python 3.10+
* AWS credentials configured with access to AgentCore Memory and Amazon Bedrock
* Amazon Bedrock model access (Claude Sonnet)

First, let's install the required libraries:

In [ ]:
!pip install "boto3>=1.42.63" "bedrock-agentcore[strands-agents]"

### Setting Up Environment

Let's import the required libraries and configure our environment:

In [ ]:
import json
import boto3
import time 
import uuid
from datetime import datetime
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.memory.client import MemoryClient

# Configuration
REGION = 'us-west-2'
MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'
STAGES = ['pending', 'shown', 'accepted', 'declined', 'applied']

# Initialize clients
agentcore_client = boto3.client('bedrock-agentcore', region_name=REGION)
bedrock_runtime = boto3.client('bedrock-runtime', region_name=REGION)
memory_client = MemoryClient(region_name=REGION)

print(f"✅ Initialized clients for region: {REGION}")

## 1. Create Memory Instance

We create a memory instance without strategies — we'll populate it externally with recommendation data rather than extracting from conversations.

In [ ]:
unique_name = f"recommendation_lifecycle_demo_{uuid.uuid4().hex[:8]}"
memory = memory_client.create_memory_and_wait(
    name=unique_name,
    strategies=[],
    description="Demo memory for managing credit card recommendation lifecycles"
)

MEMORY_ID = memory['memoryId']
print(f"✅ Created memory: {MEMORY_ID}")

## 2. Populate with Sample Data

We create recommendations at different lifecycle stages to simulate a real system. Each record is placed in a namespace that reflects its current state.

In a traditional system, this data would go into a database, then ETL to a warehouse, then a BI tool queries it (hours of lag). Here, writing to a namespace makes it instantly available for agents and analytics.

In [ ]:
sample_data = [
    {'customer_id': 'customer_001', 'stage': 'pending', 'product': 'Travel Rewards Card', 
     'reason': 'High travel spending: $2,400/month on flights and hotels'},
    {'customer_id': 'customer_001', 'stage': 'shown', 'product': 'Cashback Plus Card',
     'reason': 'Presented in previous session - 3% cashback on groceries'},
    {'customer_id': 'customer_002', 'stage': 'accepted', 'product': 'Premium Rewards',
     'reason': 'Customer expressed strong interest - high income segment'},
    {'customer_id': 'customer_003', 'stage': 'declined', 'product': 'Student Card',
     'reason': 'Customer declined due to $95 annual fee'},
    {'customer_id': 'customer_004', 'stage': 'applied', 'product': 'Business Rewards',
     'reason': 'Application submitted - awaiting approval'},
    {'customer_id': 'customer_005', 'stage': 'shown', 'product': 'Premium Rewards',
     'reason': 'Presented today - luxury segment'},
    {'customer_id': 'customer_006', 'stage': 'declined', 'product': 'Premium Rewards',
     'reason': 'Customer declined - prefers no annual fee cards'},
    {'customer_id': 'customer_007', 'stage': 'accepted', 'product': 'Travel Rewards Card',
     'reason': 'Customer interested - frequent traveler'},
]

print(f"📊 Preparing {len(sample_data)} sample recommendations across lifecycle stages")

In [ ]:
records = []
current_time = datetime.now().timestamp()

for idx, rec in enumerate(sample_data):
    namespace = f"/bank/customers/{rec['customer_id']}/recommendations/{rec['stage']}"
    
    record_data = {
        'recommendation_id': f"rec_{idx:03d}",
        'customer_id': rec['customer_id'],
        'product': rec['product'],
        'reason': rec['reason'],
        'stage': rec['stage'],
        'timestamp': current_time + idx,
        'history': [{
            'namespace': rec['stage'],
            'timestamp': current_time + idx,
            'action': f"moved_to_{rec['stage']}"
        }]
    }
    
    records.append({
        'requestIdentifier': f"{rec['customer_id']}_{rec['stage']}_{idx}",
        'namespaces': [namespace],
        'content': {'text': json.dumps(record_data)},
        'timestamp': current_time + idx
    })

response = agentcore_client.batch_create_memory_records(
    memoryId=MEMORY_ID,
    records=records
)

print(f"✅ Created {len(response['successfulRecords'])} recommendation records")
print(f"   Stages: pending, shown, accepted, declined, applied")

## 3. Demonstrate State Transition

Let's move a recommendation through the lifecycle: `pending` → `shown`.

The pattern is:
1. Read the original record from the source namespace
2. Enrich with transition metadata (timestamp, action)
3. Create in the target namespace
4. Delete from the source namespace

The record's `history` array preserves a complete audit trail of every transition.

In [ ]:
time.sleep(120) 
session_manager = MemorySessionManager(memory_id=MEMORY_ID, region_name=REGION)

# Get a pending recommendation for customer_001 (exact namespace, no wildcard needed)
pending_records = session_manager.list_long_term_memory_records(
    namespace_prefix="/bank/customers/customer_001/recommendations/pending",
    max_results=1
)

if pending_records:
    original_record_id = pending_records[0]['memoryRecordId']
    print(f"📋 Found pending recommendation: {original_record_id}")
else:
    print("⚠️ No pending recommendations found for customer_001")

In [ ]:
if pending_records:
    record = session_manager.get_memory_record(original_record_id)
    content_data = json.loads(record['content']['text'])
    
    # Enrich with transition metadata
    transition_time = datetime.now().timestamp()
    content_data['shown_timestamp'] = transition_time
    content_data['stage'] = 'shown'
    content_data['history'].append({
        'namespace': 'shown',
        'timestamp': transition_time,
        'action': 'presented_to_customer'
    })
    
    # Create in new namespace
    response = agentcore_client.batch_create_memory_records(
        memoryId=MEMORY_ID,
        records=[{
            'requestIdentifier': f"customer_001_shown_{int(transition_time)}",
            'namespaces': ['/bank/customers/customer_001/recommendations/shown'],
            'content': {'text': json.dumps(content_data)},
            'timestamp': transition_time
        }]
    )
    new_record_id = response['successfulRecords'][0]['memoryRecordId']
    
    # Delete from old namespace
    agentcore_client.delete_memory_record(
        memoryId=MEMORY_ID,
        memoryRecordId=original_record_id
    )
    
    print(f"✅ Transition complete: PENDING → SHOWN")
    print(f"   Old record: {original_record_id}")
    print(f"   New record: {new_record_id}")
    print(f"   History entries: {len(content_data['history'])}")

## 4. Product-Level Analytics

Now let's query declined recommendations to see which products are underperforming. We retrieve all records using a broad namespace prefix and filter by stage from the record content.

In [ ]:
all_records = session_manager.list_long_term_memory_records(
    namespace_prefix="/bank/customers/",
    max_results=500
)

parsed_records = [json.loads(r['content']['text']) for r in all_records]
declined_records = [r for r in parsed_records if r['stage'] == 'declined']

print(f"🔍 DECLINED RECOMMENDATIONS ANALYSIS\n")
print(f"Total declined: {len(declined_records)}\n")

product_declines = {}
for rec in declined_records:
    product_declines.setdefault(rec['product'], []).append(rec['reason'])

for product, reasons in product_declines.items():
    print(f"  {product}: {len(reasons)} decline(s)")
    for reason in reasons:
        print(f"    - {reason}")
    print()

## 5. Natural Language Insights with LLM

The real power: combine namespace queries with LLM analysis for natural language insights. No SQL required, no BI tool configuration, no dashboard building — just ask a question about the data.

In [ ]:
decline_analysis_data = [
    {'product': r['product'], 'reason': r['reason'], 'customer_id': r['customer_id']}
    for r in declined_records
]

prompt = f"""Analyze these declined credit card recommendations and provide actionable insights:

{json.dumps(decline_analysis_data, indent=2)}

Please provide:
1. Key patterns in why customers decline
2. Product-specific issues
3. Recommendations for improving acceptance rates

Be concise and actionable."""

response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": prompt}]}]
)

insights = response['output']['message']['content'][0]['text']

print("🤖 LLM INSIGHTS\n")
print(insights)

## 6. Customer-Specific Journey View

Query all recommendations for a specific customer to see their complete journey. Since we know the customer ID, we query their namespace prefix directly — it covers all stages underneath.

In [ ]:
customer_id = 'customer_001'

customer_records = session_manager.list_long_term_memory_records(
    namespace_prefix=f"/bank/customers/{customer_id}/recommendations/",
    max_results=100
)

print(f"👤 CUSTOMER JOURNEY: {customer_id}\n")
print(f"Total recommendations: {len(customer_records)}\n")

for record in customer_records:
    content = json.loads(record['content']['text'])
    print(f"  [{content['stage'].upper()}] {content['product']}")
    print(f"    Reason: {content['reason']}")
    print(f"    History: {len(content['history'])} transition(s)")
    print()

## Summary

### What We Can Answer Now

With namespace-based state management, we can instantly answer:

- **"How many pending recommendations exist for a customer?"** → Query their pending namespace
- **"Which products get declined most?"** → Retrieve all records, filter by stage, group by product
- **"What's a customer's full recommendation journey?"** → Query their namespace prefix
- **"Why are customers declining?"** → LLM analyzes declined records

All in real-time, no SQL, no data warehouse, no BI tools.

### What We Still Can't Answer

More complex questions require additional capabilities:

- ❌ "Which customers would be best suited for our Travel Rewards card?"
- ❌ "What new products should we launch based on spending patterns?"
- ❌ "Which customer segments have highest lifetime value?"

These require access to operational data (transactions, spending patterns), dynamic code generation for complex calculations, and LLM reasoning over analytical results.

**→ Continue to Part 2** to add these capabilities with code generation and multi-source reasoning.

## 7. Cleanup (Optional)

When you're done experimenting, clean up the resources created in this tutorial:

In [ ]:
try:
    memory_client.delete_memory_and_wait(memory_id=MEMORY_ID)
    print(f"✅ Deleted memory resource: {MEMORY_ID}")
except Exception as e:
    print(f"❌ Error deleting memory: {e}")